<a href="https://colab.research.google.com/github/Meemansha-spec/EDA-with-Pandas/blob/main/Churn_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('telecom_customer_churn.csv')

In [5]:
df.head()

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [6]:
#find the missing values
df.isna().sum()

,0
Customer ID,0
Gender,0
Age,0
Married,0
Number of Dependents,0
City,0
Zip Code,0
Latitude,0
Longitude,0
Number of Referrals,0


In [7]:
## remove spaces and covert the columns into lowercase
df.columns = df.columns.str.lower().str.replace(r'\s+', '_', regex=True)

In [8]:
df.head()

,customer_id,gender,age,married,number_of_dependents,city,zip_code,latitude,longitude,number_of_referrals,...,payment_method,monthly_charge,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,customer_status,churn_category,churn_reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


# Clean missing values.
## Offer
- Missing offer means the customer did not receive any offer

In [9]:
df['offer'] = df['offer'].fillna('NO Offer')

In [10]:
df['offer'].value_counts(dropna=False)

,count
offer,
NO Offer,3877
Offer B,824
Offer E,805
Offer D,602
Offer A,520
Offer C,415


## Phone service related missing values

 - If phone_service = "No", then:
 - multiple_lines should be "No Phone Service"
- avg_monthly_long_distance_charges should be 0

In [11]:
phone_no_mask = df['phone_service'].eq('No')  ## all false values are selected
df.loc[phone_no_mask & df['multiple_lines'].isna(),'multiple_lines'] = 'No Phone Service'  ## select the places where multiple lines are missing where phone service is not there, with no phone service

df.loc[phone_no_mask & df['avg_monthly_long_distance_charges'].isna(),'avg_monthly_long_distance_charges'] = 0.0

In [12]:
# Validate phone-related columns

df[["phone_service", "multiple_lines", "avg_monthly_long_distance_charges"]].isna().sum()

,0
phone_service,0
multiple_lines,0
avg_monthly_long_distance_charges,0


# Internet service related missing values

- if internet service = 'No',
- means customer does not have internet service

In [13]:
internet_no_mask = df['internet_service'].eq('No')
internet_columns = [ "internet_type",
    "online_security", "online_backup",
    "device_protection_plan", "premium_tech_support", "streaming_tv",
    "streaming_movies", "streaming_music", "unlimited_data"]

for col in internet_columns:
    df.loc[internet_no_mask & df[col].isna(), col] = "No Internet Service"


df.loc[internet_no_mask & df["avg_monthly_gb_download"].isna(), "avg_monthly_gb_download"] = 0.0
print("Internet-service-related missing values cleaned for customers without internet service.")

Internet-service-related missing values cleaned for customers without internet service.


In [14]:
df[internet_columns].isna().sum()

,0
internet_type,0
online_security,0
online_backup,0
device_protection_plan,0
premium_tech_support,0
streaming_tv,0
streaming_movies,0
streaming_music,0
unlimited_data,0


In [15]:
df['avg_monthly_gb_download'].isna().sum()

np.int64(0)

## Clean Churn Category and churn reason
- If customer_status is not `churned` , missing churn category means `not_churned`

In [16]:
non_churn_mask = df['customer_status'].ne('Churned')  ## finds all values which are not churned

df.loc[non_churn_mask & df['churn_category'].isna(), 'churn_category'] = 'Not Churned'
df.loc[non_churn_mask & df['churn_reason'].isna(), 'churn_reason'] = 'Not Churned'


In [17]:
df[['customer_status','churn_category']].isna().sum()

,0
customer_status,0
churn_category,0


In [18]:
df.isna().sum()

,0
customer_id,0
gender,0
age,0
married,0
number_of_dependents,0
city,0
zip_code,0
latitude,0
longitude,0
number_of_referrals,0


In [19]:
# Convert ID and zip code to string
# They are identifiers, not mathematical numbers.

df["customer_id"] = df["customer_id"].astype("string")
df["zip_code"] = df["zip_code"].astype("string")

In [20]:
# Convert integer columns

integer_cols = [
    "age",
    "number_of_dependents",
    "number_of_referrals",
    "tenure_in_months",
    "total_extra_data_charges"
]

for col in integer_cols:
    df[col] = df[col].astype("int64")

In [21]:
# Convert float columns

float_cols = [
    "latitude",
    "longitude",
    "avg_monthly_long_distance_charges",
    "avg_monthly_gb_download",
    "monthly_charge",
    "total_charges",
    "total_refunds",
    "total_long_distance_charges",
    "total_revenue"
]

for col in float_cols:
    df[col] = df[col].astype("float64")

In [22]:
# Check negative values in numerical columns

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

negative_value_summary = {}

for col in numerical_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        negative_value_summary[col] = negative_count

negative_value_summary


{'longitude': np.int64(7043), 'monthly_charge': np.int64(120)}

## Create business flag for analysis

In [23]:
# Customer status flags

df["is_churned"] = np.where(df["customer_status"].eq("Churned"), 1, 0)
df["is_stayed"] = np.where(df["customer_status"].eq("Stayed"), 1, 0)
df["is_joined"] = np.where(df["customer_status"].eq("Joined"), 1, 0)

# Existing customer flag
# Joined customers are new customers, so they should be treated carefully in churn calculations.

df["is_existing_customer"] = np.where(df["customer_status"].ne("Joined"), 1, 0)

In [24]:
# Service flags

df["has_phone_service"] = np.where(df["phone_service"].eq("Yes"), 1, 0)
df["has_internet_service"] = np.where(df["internet_service"].eq("Yes"), 1, 0)

# Offer flag

df["has_offer"] = np.where(df["offer"].ne("No Offer"), 1, 0)

# Contract flag

df["is_month_to_month"] = np.where(df["contract"].eq("Month-to-Month"), 1, 0)

# Negative monthly charge flag

df["monthly_charge_is_negative"] = np.where(df["monthly_charge"] < 0, 1, 0)

In [25]:
# Validate new flag columns

flag_cols = [
    "is_churned",
    "is_stayed",
    "is_joined",
    "is_existing_customer",
    "has_phone_service",
    "has_internet_service",
    "has_offer",
    "is_month_to_month",
    "monthly_charge_is_negative"
]

df[flag_cols].head()

,is_churned,is_stayed,is_joined,is_existing_customer,has_phone_service,has_internet_service,has_offer,is_month_to_month,monthly_charge_is_negative
0,0,1,0,1,1,1,1,0,0
1,0,1,0,1,1,1,1,1,1
2,1,0,0,1,1,1,1,1,0
3,1,0,0,1,1,1,1,1,0
4,1,0,0,1,1,1,1,1,0


## Customer groups

In [26]:
# Tenure group

df["tenure_group"] = pd.cut(
    df["tenure_in_months"],
    bins=[0, 6, 12, 24, 36, 48, 60, 72],
    labels=[
        "0-6 Months",
        "7-12 Months",
        "13-24 Months",
        "25-36 Months",
        "37-48 Months",
        "49-60 Months",
        "61-72 Months"
    ],
    include_lowest=True
).astype("string")

In [27]:
# Age group

df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 35, 45, 55, 65, 80],
    labels=[
        "19-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "66-80"
    ],
    include_lowest=True
).astype("string")

In [28]:
# Monthly charge group

df["monthly_charge_group"] = pd.cut(
    df["monthly_charge"],
    bins=[-np.inf, 25, 50, 75, 100, np.inf],
    labels=[
        "<=25",
        "26-50",
        "51-75",
        "76-100",
        "100+"
    ]
).astype("string")

In [29]:
# Revenue value segment based on total revenue quartiles

df["revenue_value_segment"] = pd.qcut(
    df["total_revenue"],
    q=4,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
        "Very High Value"
    ]
).astype("string")

In [30]:
# Check newly created grouping columns

df[
    [
        "customer_id",
        "age",
        "age_group",
        "tenure_in_months",
        "tenure_group",
        "monthly_charge",
        "monthly_charge_group",
        "total_revenue",
        "revenue_value_segment"
    ]
].head()

,customer_id,age,age_group,tenure_in_months,tenure_group,monthly_charge,monthly_charge_group,total_revenue,revenue_value_segment
0,0002-ORFBO,37,36-45,9,7-12 Months,65.6,51-75,974.81,Medium Value
1,0003-MKNFE,46,46-55,9,7-12 Months,-4.0,<=25,610.28,Medium Value
2,0004-TLHLJ,50,46-55,4,0-6 Months,73.9,51-75,415.45,Low Value
3,0011-IGKFF,78,66-80,13,13-24 Months,98.0,76-100,1599.51,Medium Value
4,0013-EXCHZ,75,66-80,3,0-6 Months,83.9,76-100,289.54,Low Value


## Vlidate business logic after cleaning

In [31]:
# Check phone service logic

phone_validation = pd.crosstab(
    df["phone_service"],
    df["multiple_lines"],
    dropna=False
)

phone_validation

multiple_lines,No,No Phone Service,Yes
phone_service,,,
No,0,682,0
Yes,3390,0,2971


In [32]:
# Check internet service logic

internet_validation = pd.crosstab(
    df["internet_service"],
    df["internet_type"],
    dropna=False
)

internet_validation

internet_type,Cable,DSL,Fiber Optic,No Internet Service
internet_service,,,,
No,0,0,0,1526
Yes,830,1652,3035,0


In [33]:
# Check churn reason logic

churn_reason_validation = pd.crosstab(
    df["customer_status"],
    df["churn_reason"],
    dropna=False
)

churn_reason_validation

churn_reason,Attitude of service provider,Attitude of support person,Competitor had better devices,Competitor made better offer,Competitor offered higher download speeds,Competitor offered more data,Deceased,Don't know,Extra data charges,Lack of affordable download/upload speed,...,Limited range of services,Long distance charges,Moved,Network reliability,Not Churned,Poor expertise of online support,Poor expertise of phone support,Price too high,Product dissatisfaction,Service dissatisfaction
customer_status,,,,,,,,,,,,,,,,,,,,,
Churned,94,220,313,311,100,117,6,130,39,30,...,37,64,46,72,0,31,12,78,77,63
Joined,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,454,0,0,0,0,0
Stayed,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,4720,0,0,0,0,0


In [34]:
# Revenue formula validation

df["calculated_total_revenue"] = (
    df["total_charges"]
    + df["total_long_distance_charges"]
    + df["total_extra_data_charges"]
    - df["total_refunds"]
)

df["revenue_difference"] = (
    df["total_revenue"] - df["calculated_total_revenue"]
).round(2)

revenue_mismatch_count = (df["revenue_difference"].abs() > 0.01).sum()

print("Revenue mismatch count:", revenue_mismatch_count)

Revenue mismatch count: 0


In [35]:
# Check revenue difference distribution

df["revenue_difference"].describe()

,revenue_difference
count,7043.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


In [36]:
# Drop temporary validation columns after checking

df = df.drop(
    columns=[
        "calculated_total_revenue",
        "revenue_difference"
    ]
)

print("Temporary validation columns removed.")

Temporary validation columns removed.


In [37]:
# Final dataset shape

print("Final cleaned dataset shape:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Final cleaned dataset shape:
Rows: 7043
Columns: 51


In [38]:
# Final missing value count

print("Final missing values:", df.isnull().sum().sum())

Final missing values: 0


## The Data is cleaned

In [39]:
# Final duplicate check

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

Duplicate rows: 0
Duplicate customer IDs: 0


In [40]:
# Final customer status distribution

df["customer_status"].value_counts()

,count
customer_status,
Stayed,4720
Churned,1869
Joined,454


In [41]:
# Final churn rate check

final_churn_rate = df["is_churned"].mean() * 100

print("Final Churn Rate:", round(final_churn_rate, 2), "%")

Final Churn Rate: 26.54 %


In [42]:
# Cleaning summary for documentation

cleaning_summary = pd.DataFrame({
    "cleaning_step": [
        "Standardized column names",
        "Removed extra spaces from text columns",
        "Checked and removed duplicate rows",
        "Checked duplicate customer IDs",
        "Filled missing offers",
        "Cleaned phone-service-related missing values",
        "Cleaned internet-service-related missing values",
        "Cleaned churn category and churn reason",
        "Corrected ID and zip code data types",
        "Created customer status flags",
        "Created service and offer flags",
        "Created tenure, age, monthly charge, and revenue groups",
        "Validated revenue formula"
    ],
    "business_reason": [
        "Makes columns easier to use in Python, SQL, and Power BI",
        "Prevents duplicate categories caused by hidden spaces",
        "Avoids inflated customer counts and revenue values",
        "Ensures one row represents one customer",
        "Missing offer means customer did not receive an offer",
        "Phone-related blanks mean no phone service, not an error",
        "Internet-related blanks mean no internet service, not an error",
        "Non-churned customers do not have churn reasons",
        "IDs should not be treated as numerical values",
        "Makes churn calculations easier in SQL and Power BI",
        "Improves segmentation and dashboard filtering",
        "Supports business-friendly analysis and dashboard slicers",
        "Ensures revenue fields are internally consistent"
    ]
})

cleaning_summary

,cleaning_step,business_reason
0,Standardized column names,"Makes columns easier to use in Python, SQL, an..."
1,Removed extra spaces from text columns,Prevents duplicate categories caused by hidden...
2,Checked and removed duplicate rows,Avoids inflated customer counts and revenue va...
3,Checked duplicate customer IDs,Ensures one row represents one customer
4,Filled missing offers,Missing offer means customer did not receive a...
5,Cleaned phone-service-related missing values,"Phone-related blanks mean no phone service, no..."
6,Cleaned internet-service-related missing values,Internet-related blanks mean no internet servi...
7,Cleaned churn category and churn reason,Non-churned customers do not have churn reasons
8,Corrected ID and zip code data types,IDs should not be treated as numerical values
9,Created customer status flags,Makes churn calculations easier in SQL and Pow...


In [45]:
df.to_csv('Telecom_cleaned.csv',index = False)

In [44]:
cleaning_summary.to_csv('cleaning_suammry.csv',index = False)

In [47]:
clean  =  pd.read_csv('Telecom_cleaned.csv')

In [48]:
clean.head()

,customer_id,gender,age,married,number_of_dependents,city,zip_code,latitude,longitude,number_of_referrals,...,is_existing_customer,has_phone_service,has_internet_service,has_offer,is_month_to_month,monthly_charge_is_negative,tenure_group,age_group,monthly_charge_group,revenue_value_segment
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,1,1,1,1,0,0,7-12 Months,36-45,51-75,Medium Value
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,1,1,1,1,1,1,7-12 Months,46-55,<=25,Medium Value
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,1,1,1,1,1,0,0-6 Months,46-55,51-75,Low Value
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,1,1,1,1,1,0,13-24 Months,66-80,76-100,Medium Value
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,1,1,1,1,1,0,0-6 Months,66-80,76-100,Low Value


In [49]:
clean.isna().sum()

,0
customer_id,0
gender,0
age,0
married,0
number_of_dependents,0
city,0
zip_code,0
latitude,0
longitude,0
number_of_referrals,0
